In [2]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

here = Path(os.getcwd())
print("Dossier courant :", here)

root = next(p for p in [here, *here.parents] if (p / "data" / "features_spx.csv").exists())
sys.path.append(str(root))
print("Racine du projet :", root)

feat = pd.read_csv(root / "data" / "features_spx.csv", index_col=0, parse_dates=True)
train = feat[feat.index < "2015-01-01"]
print(feat.shape, train.shape)

Dossier courant : C:\Users\Syrine\Documents\GitHub\ml-finance-project
Racine du projet : C:\Users\Syrine\Documents\GitHub\ml-finance-project
(6690, 9) (3751, 9)


In [3]:
X = sm.add_constant(train[["rv_d", "rv_w", "rv_m"]])

for h in [1, 5, 22]:
    res = sm.OLS(train[f"y_h{h}"], X).fit(cov_type="HAC", cov_kwds={"maxlags": h})
    print(f"--- h = {h} ---")
    print(res.summary().tables[1])

--- h = 1 ---
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1211      0.060      2.020      0.043       0.004       0.238
rv_d           0.1009      0.068      1.487      0.137      -0.032       0.234
rv_w           0.6158      0.166      3.707      0.000       0.290       0.941
rv_m           0.1588      0.141      1.128      0.259      -0.117       0.435
--- h = 5 ---
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1687      0.064      2.653      0.008       0.044       0.293
rv_d           0.1166      0.062      1.881      0.060      -0.005       0.238
rv_w           0.4908      0.118      4.161      0.000       0.260       0.722
rv_m           0.2197      0.150      1.464      0.143      -0.074       0.514
--- h = 22 ---
         

In [4]:
rows = []
for name in ["spx", "tsx"]:
    feat = pd.read_csv(root / "data" / f"features_{name}.csv", index_col=0, parse_dates=True)
    train = feat[feat.index < "2015-01-01"]
    X = sm.add_constant(train[["rv_d", "rv_w", "rv_m"]])
    for h in [1, 5, 22]:
        res = sm.OLS(train[f"y_h{h}"], X).fit(cov_type="HAC", cov_kwds={"maxlags": h})
        row = {"indice": name, "horizon": h}
        for v in X.columns:
            row[f"coef_{v}"] = res.params[v]
            row[f"se_{v}"] = res.bse[v]
        row["somme_betas"] = res.params[["rv_d", "rv_w", "rv_m"]].sum()
        row["R2"] = res.rsquared
        row["n_train"] = int(res.nobs)
        rows.append(row)

table2 = pd.DataFrame(rows)
(root / "results").mkdir(exist_ok=True)
table2.round(4).to_csv(root / "results" / "table2_har.csv", index=False)
display(table2.round(3))

,indice,horizon,coef_const,se_const,coef_rv_d,se_rv_d,coef_rv_w,se_rv_w,coef_rv_m,se_rv_m,somme_betas,R2,n_train
0,spx,1,0.121,0.060,0.101,0.068,0.616,0.166,0.159,0.141,0.876,0.425,3751
1,spx,5,0.169,0.064,0.117,0.062,0.491,0.118,0.220,0.150,0.827,0.599,3751
2,spx,22,0.303,0.054,0.069,0.024,0.273,0.096,0.347,0.107,0.689,0.507,3751
3,tsx,1,0.126,0.077,0.089,0.101,0.657,0.314,0.112,0.187,0.858,0.382,3737
4,tsx,5,0.177,0.080,0.090,0.082,0.601,0.300,0.110,0.179,0.801,0.592,3737
5,tsx,22,0.311,0.070,0.069,0.026,0.249,0.133,0.333,0.071,0.651,0.454,3737


## Interprétation du HAR (entraînement 2000-2014)

- **Déplacement du poids** : à h = 1, le poids principal est sur la volatilité de la semaine (rv_w : 0,62 pour le S&P 500, 0,66 pour le TSX). Quand l'horizon passe à 22 jours, le poids se déplace vers la volatilité du mois (rv_m : 0,35 et 0,33), alors que rv_w tombe à 0,27 et 0,25.
- **Persistance** : la somme des trois coefficients est élevée (0,86-0,88 à h = 1, 0,65-0,69 à h = 22) : la volatilité est persistante. Elle est plus faible à 22 jours car, sur un mois, la volatilité revient vers sa moyenne.
- **Significativité** : avec les écarts-types HAC, seul rv_w est significatif à h = 1. À h = 22 les trois variables le sont pour le S&P 500.
- **Qualité** : R² entre 0,38 et 0,60. Le modèle explique une bonne partie de la volatilité future avec seulement 3 variables.
- **Limite** : le HAR est estimé sur 2000-2014 seulement. Le test 2015-2026 reste intact pour comparer les modèles aux sprints suivants.

In [5]:
from src.utils import TEST_START, save_forecasts

HAR_COLS = ["rv_d", "rv_w", "rv_m"]

def har_walk_forward(feat, h, refit=22):
    test_dates = feat.index[feat.index >= TEST_START]
    out, res = [], None
    for k, d in enumerate(test_dates):
        if k % refit == 0:
            train = feat.loc[:d].iloc[:-h]
            res = sm.OLS(train[f"y_h{h}"], sm.add_constant(train[HAR_COLS])).fit()
        x = sm.add_constant(feat.loc[[d], HAR_COLS], has_constant="add")
        out.append({"date": d, "model": "HAR", "horizon": h,
                    "forecast": max(float(res.predict(x).iloc[0]), 1e-4),
                    "actual": feat.loc[d, f"y_h{h}"]})
    return pd.DataFrame(out)

feat = pd.read_csv(root / "data" / "features_spx.csv", index_col=0, parse_dates=True)
df = pd.concat([har_walk_forward(feat, h) for h in [1, 5, 22]])
save_forecasts(df, "HAR", "spx")
print(df.shape)

(8817, 5)


In [6]:
from src.utils import qlike, mse
for h, g in df.groupby("horizon"):
    print(f"h = {h}  QLIKE = {qlike(g.actual, g.forecast):.4f}  MSE = {mse(g.actual, g.forecast):.4f}")

h = 1  QLIKE = 0.3960  MSE = 1.3104
h = 5  QLIKE = 0.2754  MSE = 0.9952
h = 22  QLIKE = 0.3127  MSE = 0.8817


In [7]:
feat_tsx = pd.read_csv(root / "data" / "features_tsx.csv", index_col=0, parse_dates=True)
df_tsx = pd.concat([har_walk_forward(feat_tsx, h) for h in [1, 5, 22]])
save_forecasts(df_tsx, "HAR", "tsx")
print(df_tsx.shape)

for h, g in df_tsx.groupby("horizon"):
    print(f"h = {h}  QLIKE = {qlike(g.actual, g.forecast):.4f}  MSE = {mse(g.actual, g.forecast):.4f}")

(8799, 5)
h = 1  QLIKE = 0.3416  MSE = 1.1488
h = 5  QLIKE = 0.2384  MSE = 0.9970
h = 22  QLIKE = 0.3418  MSE = 0.9754


In [8]:
import importlib, src.utils
importlib.reload(src.utils)
from src.utils import evaluate

print(evaluate("spx"))
print(evaluate("tsx"))

   horizon model   QLIKE     MSE  DM vs HAR  p-value
0        1   HAR  0.3960  1.3104        NaN      NaN
1        5   HAR  0.2754  0.9952        NaN      NaN
2       22   HAR  0.3127  0.8817        NaN      NaN
   horizon model   QLIKE     MSE  DM vs HAR  p-value
0        1   HAR  0.3416  1.1488        NaN      NaN
1        5   HAR  0.2384  0.9970        NaN      NaN
2       22   HAR  0.3418  0.9754        NaN      NaN
